<style>
/deep/ .rendered_html, .rendered_html { font-family: Arial, Helvetica, sans-serif !important; }
h1, h2, h3, h4, h5, h6 { font-family: Arial, Helvetica, sans-serif !important; }
</style>
# Interacti<span style="color:#c62828; font-weight:bold; font-size:1.1em">V</span>e Monte Carlo Notebook

This interactive notebook demonstrates Monte Carlo simulations with live controls (ipywidgets). All rendered text in this notebook uses a sans-serif font (Arial/Helvetica) to match your request. The letter 'V' in the title is emphasized as requested.


## Instructions
Use the controls below to simulate Geometric Brownian Motion (GBM), estimate expectations, and price a European call. Set parameters and press the **Run** button. Use the Save options to export plot (PNG) or simulation data (NPZ).


In [ ]:
# Imports and display settings
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
from monte_carlo import simulate_gbm, price_european_call_mc
import os

# Ensure output directory exists
os.makedirs('notebooks/output', exist_ok=True)

# Apply inline CSS to ensure sans-serif for widget labels too
display(HTML('<style> label, .widget-label, .widget-readout { font-family: Arial, Helvetica, sans-serif !important; } </style>'))
print('Setup complete')


In [ ]:
# Widget definitions
S0 = widgets.FloatText(value=100.0, description='S0:', step=1.0)
mu = widgets.FloatText(value=0.05, description='mu:')
sigma = widgets.FloatText(value=0.2, description='sigma:')
T = widgets.FloatText(value=1.0, description='T:')
steps = widgets.IntSlider(value=252, min=1, max=2000, step=1, description='steps:')
n_paths = widgets.IntSlider(value=500, min=1, max=20000, step=1, description='n_paths:')
n_plot = widgets.IntSlider(value=200, min=0, max=2000, step=10, description='plot_paths:')
plot_display = widgets.Checkbox(value=True, description='Show plot')
save_plot = widgets.Text(value='notebooks/output/interactive_gbm.png', description='Save plot:')
save_data = widgets.Text(value='notebooks/output/interactive_gbm.npz', description='Save data:')
run_button = widgets.Button(description='Run', button_style='success')
out = widgets.Output(layout=widgets.Layout(border='1px solid lightgray'))

controls_row1 = widgets.HBox([S0, mu, sigma, T])
controls_row2 = widgets.HBox([steps, n_paths, n_plot])
controls_row3 = widgets.HBox([plot_display, save_plot, save_data])
ui = widgets.VBox([controls_row1, controls_row2, controls_row3, run_button, out])
display(ui)


In [ ]:
def run_simulation(b):
    with out:
        clear_output(wait=True)
        print('Running simulation...')
        # simulate
        times, paths = simulate_gbm(float(S0.value), float(mu.value), float(sigma.value), float(T.value), int(steps.value), int(n_paths.value))
        mean_path = paths.mean(axis=0)
        print(f'Simulated {paths.shape[0]} paths with {paths.shape[1]-1} steps')
        # plotting
        if plot_display.value:
            fig, ax = plt.subplots(figsize=(9,6), dpi=100)
            # plot subset to avoid overcrowding
            to_plot = min(int(n_plot.value), paths.shape[0])
            for i in range(to_plot):
                ax.plot(times, paths[i], color='gray', alpha=0.4, linewidth=0.6)
            ax.plot(times, mean_path, color='red', linewidth=2, label='mean')
            ax.set_xlabel('Time')
            ax.set_ylabel('S')
            ax.set_title('GBM sample paths')
            ax.legend()
            plt.show()
        # save plot if requested
        if save_plot.value:
            save_path = save_plot.value
            if save_path and not os.path.splitext(save_path)[1]:
                save_path = save_path + '.png'
            fig.savefig(save_path, dpi=100, bbox_inches='tight')
            print(f'Saved plot to {save_path}')
        # save data if requested
        if save_data.value:
            data_path = save_data.value
            if data_path and not os.path.splitext(data_path)[1]:
                data_path = data_path + '.npz'
            np.savez_compressed(data_path, times=times, paths=paths)
            print(f'Saved data to {data_path}')
        # price a sample option using same params (short demonstration)
        price, se = price_european_call_mc(float(S0.value), float(S0.value), 0.05, float(sigma.value), float(T.value), 50000, antithetic=True)
        print(f'Example European call (S0=K) price ≈ {price:.6f} (SE={se:.6f})')

run_button.on_click(run_simulation)
print('Ready — change parameters and press Run')
